In [0]:
import sys
from typing import Final

_CONFIG_PATH: Final[str] = "../../00_setup/pawel_project"

if _CONFIG_PATH not in sys.path:
    sys.path.insert(0, _CONFIG_PATH)

In [0]:
from music_pipeline_setup import music_stats_tables


silver_music_tbl_path: str = music_stats_tables["silver"]
gold_music_tbl_path: str = music_stats_tables["gold"]

In [0]:
print(gold_music_tbl_path)

In [0]:
# Plans to do:
# Aggregate silver table; specifically:
# sum number of views/comments/likes per artist per timestamp
# snapshots of views/comments/likes per song 
# there's gonna be two golden tables:
# 1. artist table
# 2. song

In [0]:
print(silver_music_tbl_path)

In [0]:
silver_table = spark.read.table(silver_music_tbl_path)
display(silver_table)

In [0]:
# First, truncate _ingested_at to minutes (no need for seconds/miliseconds etc.)
from pyspark.sql.functions import date_trunc, col

truncated_date = date_trunc("minute", col("_ingested_at"))

silver_table_minutes = silver_table.withColumn("_ingested_at_minutes", truncated_date)

In [0]:
display(silver_table_minutes)

# First golden layer - sum of views/comments/like per author per hour

In [0]:
from pyspark.sql.functions import sum, max, min, mean, std, countDistinct, round

gold_table_grouped_by_author_and_timestamp = silver_table_minutes.groupBy("author", "_ingested_at_minutes").agg(
    countDistinct("video_id").alias("total_videos"),
    sum("view_count").alias("total_views"),
    sum("like_count").alias("total_likes"),
    sum("comment_count").alias("total_comments"),
    max("view_count").alias("max_views"),
    max("like_count").alias("max_likes"),
    max("comment_count").alias("max_comments"),
    min("view_count").alias("min_views"),
    min("like_count").alias("min_likes"),
    min("comment_count").alias("min_comments"),
    round(mean("view_count"), 1).alias("mean_views"),
    round(mean("like_count"), 1).alias("mean_likes"),
    round(mean("comment_count"), 1).alias("mean_comments"),
    round(100*std("view_count") / mean("view_count"), 2).alias("cv_views_pct"),
    round(100*std("like_count") / mean("like_count"), 2).alias("cv_likes_pct"),
    round(100*std("comment_count") / mean("comment_count"), 2).alias("cv_comments_pct"),
)

In [0]:
tbl_1_name = gold_music_tbl_path+"_by_author"
gold_table_grouped_by_author_and_timestamp.write.format("delta").mode("append").saveAsTable(tbl_1_name)


In [0]:
display(gold_table_grouped_by_author_and_timestamp.show(5))

In [0]:
gold_table_grouped_by_video_and_timestamp = silver_table_minutes.groupBy("author", "song_title", "_ingested_at_minutes").agg(
    sum("view_count").alias("total_views"),
    sum("like_count").alias("total_likes"),
    sum("comment_count").alias("total_comments")
)

In [0]:
tbl_2_name = gold_music_tbl_path+"_by_video"
gold_table_grouped_by_author_and_timestamp.write.format("delta").mode("append").saveAsTable(tbl_2_name)


In [0]:
display(gold_table_grouped_by_video_and_timestamp.show(5))